# Machine Unlearning Experiments - Fixed Implementation

This notebook runs all unlearning experiments using refactored scripts:
- `main.py` - Standard MIA experiments (entropy-based logistic regression)
- `main_new_metrics.py` - Advanced MIA experiments (LiRA, Quantile, Shadow via forget_new_metrics.py)
- `train_model.py` - Baseline model training
- `convert_csv.py` - Compile results to CSV
- `calculate_miau_per_seed.py` - Calculate MIAU scores

## Method Configurations
Experiments run with ret_perc support:
- `baseline`, `retrain` (ret_perc=0, gold standard)
- `retrain25`, `retrain50`, `retrain75` (partial forgetting)
- `finetune`, `teacher`, `amnesiac`, `ssdtuning`

## Fixes Applied:
- **datasets.py**: Fixed augmentation control, list mutation bug, MUCAC split inversion
- **metrics.py**: Fixed class-biased sampling, hardcoded random_state
- **forget.py**: Fixed model reference bug in retrain
- **calculate_miau_per_seed.py**: Fixed F_i formula for ideal unlearning
- **metrics_mia.py**: Fixed LiRA label inversion, shadow perc=0 issue

## Setup Instructions
1. Set `BASE_PATH` below to your Google Drive folder
2. Select GPU runtime: `Runtime → Change runtime type → GPU`
3. Run all cells: `Runtime → Run all`

In [ ]:
# ============ CONFIGURE THIS ONCE ============
BASE_PATH = "/content/drive/MyDrive/DeepUnlearning/Unlearning/source_submitted"
# =============================================

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir(BASE_PATH)
print(f"Working directory: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install xgboost scikit-learn pandas numpy matplotlib tqdm transformers

In [ ]:
# Imports and setup
import json
import os
import subprocess
import sys
import os
os.environ['PYTHONUNBUFFERED'] = '1'

sys.path.insert(0, BASE_PATH)

# Load configuration
with open(os.path.join(BASE_PATH, 'config.json'), 'r') as f:
    CONFIG = json.load(f)

SEEDS = CONFIG['seeds']
OUTPUT_BASE = CONFIG['output_base']
# Use method_configs which includes ret_perc for retrain25/50/75 experiments
METHOD_CONFIGS = CONFIG.get('method_configs', [
    {"name": m, "method": m, "ret_perc": 0} for m in CONFIG.get('methods', [])
])

# Create output directory
os.makedirs(OUTPUT_BASE, exist_ok=True)
print(f"Output directory: {OUTPUT_BASE}")
print(f"Seeds: {SEEDS}")
print(f"Method configs: {[m['name'] for m in METHOD_CONFIGS]}")
print(f"Number of experiments: {len(CONFIG['experiments'])}")

In [ ]:
# ========== HELPER FUNCTIONS ==========

def run_command_realtime(cmd):
    """
    Run a command with real-time output streaming.
    Uses Popen to stream stdout line by line.
    """
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    return process.returncode


def resolve_retrain_folder(rel_or_abs):
    """
    Locate a retrain-model folder. Tries BASE_PATH first, then its parent
    (where Retrain Models often lives next to source_submitted).
    """
    if not rel_or_abs:
        return None
    if os.path.isabs(rel_or_abs) and os.path.exists(rel_or_abs):
        return rel_or_abs
    candidates = [
        os.path.join(BASE_PATH, rel_or_abs),
        os.path.join(os.path.dirname(BASE_PATH.rstrip("/")), rel_or_abs),
        rel_or_abs,
    ]
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return candidates[0]


def run_main_py(exp_config, seed, method_config):
    """
    Run main.py with the given configuration.
    
    method_config: dict with 'name', 'method', 'ret_perc' keys
        e.g., {"name": "retrain25", "method": "retrain", "ret_perc": 25}
    
    Returns: 0 if skipped (already exists), returncode otherwise
    """
    output_dir = os.path.join(OUTPUT_BASE, f"Results_{exp_config['name']}")
    os.makedirs(output_dir, exist_ok=True)
    
    # Support both old-style string method and new-style dict method_config
    if isinstance(method_config, str):
        method_name = method_config
        method_cmd = method_config
        ret_perc = 0
    else:
        method_name = method_config['name']
        method_cmd = method_config['method']
        ret_perc = method_config.get('ret_perc', 0)
    
    # Check if result file already exists (resume capability)
    output_file = f"{method_name}_{exp_config['dataset'].lower()}_{exp_config['model'].lower()}_seed_{seed}.txt"
    output_path = os.path.join(output_dir, output_file)
    if os.path.exists(output_path):
        print(f"⏭️ Skipping {output_file} - already exists")
        return 0
    
    cmd = [
        "python", "-u", "main.py",
        "-net", exp_config['model'],
        "-dataset", exp_config['dataset'],
        "-classes", str(exp_config['num_classes']),
        "-weight_path", os.path.join(BASE_PATH, exp_config['baseline_path']),
        "-csv_path", os.path.join(BASE_PATH, exp_config['split_csv']),
        "-forget_per_class", str(exp_config['forget_per_class']),
        "-method", method_cmd,
        "-ret_perc", str(ret_perc),
        "-seed", str(seed),
        "-b", str(exp_config.get('batch_size', 256)),
        "-output_dir", output_dir,
        "-gpu",
    ]
    
    retrain_folder = resolve_retrain_folder(exp_config.get("retrain_folder"))
    if retrain_folder:
        cmd.extend(["-retrain_folder", retrain_folder])
    
    # Add augmentation flag if needed
    if not exp_config.get('use_augmentation', True):
        cmd.append("-no_augmentation")
    
    # Add epochs override for underfitted
    if 'epochs_retrain' in exp_config:
        cmd.extend(["-epochs_override", str(exp_config['epochs_retrain'])])
    if 'milestones' in exp_config:
        cmd.extend(["-milestones_override", ','.join(map(str, exp_config['milestones']))])
    
    print(f"\n{'='*60}")
    print(f"Running: {exp_config['name']} | Seed: {seed} | Method: {method_name}")
    print(f"Command: {' '.join(cmd)}")
    print(f"{'='*60}", flush=True)
    
    return run_command_realtime(cmd)


def run_train_model(model_name, dataset_name, num_classes, epochs, milestones,
                    use_augmentation, output_path, batch_size=256, seed=None):
    """
    Run train_model.py to train a baseline model.
    """
    cmd = [
        "python", "-u", "train_model.py",
        "-net", model_name,
        "-dataset", dataset_name,
        "-classes", str(num_classes),
        "-epochs_override", str(epochs),
        "-milestones_override", ','.join(map(str, milestones)),
        "-b", str(batch_size),
        "-output_path", output_path,
        "-gpu",
    ]
    
    if not use_augmentation:
        cmd.append("-no_augmentation")
    
    if seed is not None:
        cmd.extend(["-seed", str(seed)])
    
    print(f"\n{'='*60}")
    print(f"Training baseline: {model_name} on {dataset_name}")
    print(f"Epochs: {epochs}, Augmentation: {use_augmentation}")
    print(f"Command: {' '.join(cmd)}")
    print(f"{'='*60}", flush=True)
    
    return run_command_realtime(cmd)


def run_convert_csv(folder_path):
    """
    Run convert_csv.py to compile results.
    """
    cmd = ["python", "-u", "convert_csv.py", "-folder", folder_path]
    
    print(f"\nCompiling results in: {folder_path}", flush=True)
    run_command_realtime(cmd)
    
    return os.path.join(folder_path, 'compiled_results.csv')


def run_calculate_miau(csv_path):
    """
    Run calculate_miau_per_seed.py on a compiled CSV.
    """
    cmd = ["python", "-u", "calculate_miau_per_seed.py", "--input", csv_path]
    
    print(f"\nCalculating MIAU for: {csv_path}", flush=True)
    return run_command_realtime(cmd)


def run_main_new_metrics(exp_config, seed, method_config):
    """
    Run main_new_metrics.py for unlearning with advanced MIA metrics.
    
    method_config: dict with 'name', 'method', 'ret_perc' keys
    
    Returns: 0 if skipped (already exists), returncode otherwise
    """
    output_dir = os.path.join(OUTPUT_BASE, f"Results_{exp_config['name']}_AdvancedMIA")
    os.makedirs(output_dir, exist_ok=True)
    
    # Support both old-style string and new-style dict
    if isinstance(method_config, str):
        method_name = method_config
        method_cmd = method_config
        ret_perc = 0
    else:
        method_name = method_config['name']
        method_cmd = method_config['method']
        ret_perc = method_config.get('ret_perc', 0)
    
    # Check if result file already exists (resume capability)
    output_file = f"{method_name}_{exp_config['dataset'].lower()}_{exp_config['model'].lower()}_seed_{seed}.txt"
    output_path = os.path.join(output_dir, output_file)
    if os.path.exists(output_path):
        print(f"⏭️ Skipping {output_file} - already exists")
        return 0
    
    cmd = [
        "python", "-u", "main_new_metrics.py",
        "-net", exp_config['model'],
        "-dataset", exp_config['dataset'],
        "-classes", str(exp_config['num_classes']),
        "-weight_path", os.path.join(BASE_PATH, exp_config['baseline_path']),
        "-csv_path", os.path.join(BASE_PATH, exp_config['split_csv']),
        "-forget_per_class", str(exp_config['forget_per_class']),
        "-method", method_cmd,
        "-ret_perc", str(ret_perc),
        "-seed", str(seed),
        "-b", str(exp_config.get('batch_size', 256)),
        "-output_dir", output_dir,
        "-gpu",
    ]
    
    retrain_folder = resolve_retrain_folder(exp_config.get("retrain_folder"))
    if retrain_folder:
        cmd.extend(["-retrain_folder", retrain_folder])
    
    print(f"\n{'='*60}")
    print(f"Running Advanced MIA: {method_name} | Seed: {seed}")
    print(f"Command: {' '.join(cmd)}")
    print(f"{'='*60}", flush=True)
    
    return run_command_realtime(cmd)

## Run Standard Experiments

These experiments use the standard MIA (entropy-based logistic regression).

In [ ]:
# ========== RUN ALL STANDARD EXPERIMENTS ==========

for exp_config in CONFIG['experiments']:
    name = exp_config['name']
    
    # Skip experiments that need baseline training (handled separately)
    if exp_config.get('needs_baseline_train', False):
        print(f"\nSkipping {name} - requires baseline training (handled separately)")
        continue
    
    print(f"\n{'#'*80}")
    print(f"# EXPERIMENT: {name}")
    print(f"{'#'*80}")
    
    for seed in SEEDS:
        for method_config in METHOD_CONFIGS:
            try:
                run_main_py(exp_config, seed, method_config)
            except Exception as e:
                print(f"ERROR: {e}")
    
    # Compile results and calculate MIAU
    output_folder = os.path.join(OUTPUT_BASE, f"Results_{name}")
    csv_path = run_convert_csv(output_folder)
    if os.path.exists(csv_path):
        run_calculate_miau(csv_path)

## Underfitted Experiment (1 Epoch)

First train the underfitted baseline using `train_model.py`, then run experiments.

In [ ]:
# ========== TRAIN UNDERFITTED BASELINE ==========

# Find underfitted config
underfitted_config = None
for exp in CONFIG['experiments']:
    if 'Underfitted' in exp['name']:
        underfitted_config = exp
        break

if underfitted_config:
    baseline_path = os.path.join(BASE_PATH, underfitted_config['baseline_path'])
    
    if not os.path.exists(baseline_path):
        print("Training underfitted baseline (1 epoch)...")
        run_train_model(
            model_name=underfitted_config['model'],
            dataset_name=underfitted_config['dataset'],
            num_classes=underfitted_config['num_classes'],
            epochs=underfitted_config['epochs_baseline'],
            milestones=underfitted_config['milestones'],
            use_augmentation=underfitted_config['use_augmentation'],
            output_path=baseline_path,
            batch_size=underfitted_config.get('batch_size', 256)
        )
    else:
        print(f"Underfitted baseline already exists: {baseline_path}")
else:
    print("No underfitted config found in config.json")

In [ ]:
# Run underfitted experiments
if underfitted_config:
    print(f"\n{'#'*80}")
    print(f"# UNDERFITTED EXPERIMENTS")
    print(f"{'#'*80}")
    
    for seed in SEEDS:
        for method_config in METHOD_CONFIGS:
            try:
                run_main_py(underfitted_config, seed, method_config)
            except Exception as e:
                print(f"ERROR: {e}")
    
    output_folder = os.path.join(OUTPUT_BASE, f"Results_{underfitted_config['name']}")
    csv_path = run_convert_csv(output_folder)
    if os.path.exists(csv_path):
        run_calculate_miau(csv_path)

## Overfitted Experiment (No Augmentation)

In [ ]:
# ========== TRAIN OVERFITTED BASELINE ==========

# Find overfitted config
overfitted_config = None
for exp in CONFIG['experiments']:
    if 'Overfitted' in exp['name']:
        overfitted_config = exp
        break

if overfitted_config:
    baseline_path = os.path.join(BASE_PATH, overfitted_config['baseline_path'])
    
    if not os.path.exists(baseline_path):
        print("Training overfitted baseline (no augmentation)...")
        run_train_model(
            model_name=overfitted_config['model'],
            dataset_name=overfitted_config['dataset'],
            num_classes=overfitted_config['num_classes'],
            epochs=overfitted_config['epochs_baseline'],
            milestones=overfitted_config['milestones'],
            use_augmentation=overfitted_config['use_augmentation'],  # False!
            output_path=baseline_path,
            batch_size=overfitted_config.get('batch_size', 256)
        )
    else:
        print(f"Overfitted baseline already exists: {baseline_path}")
else:
    print("No overfitted config found in config.json")

In [ ]:
# Run overfitted experiments
if overfitted_config:
    print(f"\n{'#'*80}")
    print(f"# OVERFITTED EXPERIMENTS")
    print(f"{'#'*80}")
    
    for seed in SEEDS:
        for method_config in METHOD_CONFIGS:
            try:
                run_main_py(overfitted_config, seed, method_config)
            except Exception as e:
                print(f"ERROR: {e}")
    
    output_folder = os.path.join(OUTPUT_BASE, f"Results_{overfitted_config['name']}")
    csv_path = run_convert_csv(output_folder)
    if os.path.exists(csv_path):
        run_calculate_miau(csv_path)

## MUCAC Experiment (Fixed Split)

The MUCAC dataset had an inverted split bug in the original code. The baseline and retrain models need to be retrained with the fixed `datasets.py`.

This section:
1. Trains a new MUCAC baseline model (if not exists)
2. Runs all unlearning experiments (retrain models will be trained from scratch since retrain_folder is empty)

In [ ]:
# ========== TRAIN MUCAC BASELINE (FIXED SPLIT) ==========

# Find MUCAC config
mucac_config = None
for exp in CONFIG['experiments']:
    if 'MUCAC' in exp['name'] and exp.get('needs_baseline_train', False):
        mucac_config = exp
        break

if mucac_config:
    baseline_path = os.path.join(BASE_PATH, mucac_config['baseline_path'])
    
    # Create checkpoint directory
    os.makedirs(os.path.dirname(baseline_path), exist_ok=True)
    
    if not os.path.exists(baseline_path):
        print("Training MUCAC baseline with FIXED split...")
        print(f"Output path: {baseline_path}")
        
        # Build the training command
        cmd = [
            "python", "-u", "train_model.py",
            "-net", mucac_config['model'],
            "-dataset", mucac_config['dataset'],
            "-classes", str(mucac_config['num_classes']),
            "-epochs_override", str(mucac_config['epochs_baseline']),
            "-milestones_override", ','.join(map(str, mucac_config['milestones'])),
            "-b", str(mucac_config.get('batch_size', 256)),
            "-output_path", baseline_path,
            "-gpu",
        ]
        
        print(f"\n{'='*60}")
        print(f"Training baseline: {mucac_config['model']} on {mucac_config['dataset']}")
        print(f"Epochs: {mucac_config['epochs_baseline']}, Augmentation: {mucac_config['use_augmentation']}")
        print(f"Command: {' '.join(cmd)}")
        print(f"{'='*60}", flush=True)
        
        # Log file path - save training output to txt file
        log_path = os.path.join(os.path.dirname(baseline_path), "mucac_training.txt")
        
        # Run and capture output to both terminal and file
        with open(log_path, 'w') as log_file:
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end='', flush=True)  # Print to terminal
                log_file.write(line)              # Write to file
            process.wait()
        
        print(f"\n{'='*60}")
        print(f"Training log saved to: {log_path}")
        print(f"{'='*60}")
    else:
        print(f"MUCAC baseline already exists: {baseline_path}")
else:
    print("No MUCAC config with needs_baseline_train=true found in config.json")

In [ ]:
# ========== RUN MUCAC EXPERIMENTS ==========
# Retrain models will be trained from scratch (no pre-existing retrain folder)

if mucac_config:
    print(f"\n{'#'*80}")
    print(f"# MUCAC EXPERIMENTS (FIXED SPLIT)")
    print(f"{'#'*80}")
    
    # Create retrain folder if needed
    retrain_folder = os.path.join(BASE_PATH, mucac_config.get('retrain_folder', 'checkpoint/mucac_fixed/retrain'))
    os.makedirs(retrain_folder, exist_ok=True)
    
    for seed in SEEDS:
        for method_config in METHOD_CONFIGS:
            try:
                run_main_py(mucac_config, seed, method_config)
            except Exception as e:
                print(f"ERROR: {e}")
    
    output_folder = os.path.join(OUTPUT_BASE, f"Results_{mucac_config['name']}")
    csv_path = run_convert_csv(output_folder)
    if os.path.exists(csv_path):
        run_calculate_miau(csv_path)
else:
    print("Skipping MUCAC experiments - no config found")

## Full Class Forgetting (electrical_devices)

In [ ]:
# ========== FULL CLASS EXPERIMENTS ==========

for exp_config in CONFIG.get('fullclass_experiments', []):
    name = exp_config['name']
    output_dir = os.path.join(OUTPUT_BASE, f"Results_{name}")
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n{'#'*80}")
    print(f"# FULL CLASS EXPERIMENT: {name}")
    print(f"# Forgetting class: {exp_config['forget_class']}")
    print(f"{'#'*80}", flush=True)
    
    for seed in SEEDS:
        for method_config in METHOD_CONFIGS:
            # Support both old-style string and new-style dict
            if isinstance(method_config, str):
                method_name = method_config
                method_cmd = method_config
                ret_perc = 0
            else:
                method_name = method_config['name']
                method_cmd = method_config['method']
                ret_perc = method_config.get('ret_perc', 0)
            
            # Check if result file already exists (resume capability)
            output_file = f"{method_name}_{exp_config['dataset'].lower()}_{exp_config['model'].lower()}_seed_{seed}.txt"
            output_path = os.path.join(output_dir, output_file)
            if os.path.exists(output_path):
                print(f"⏭️ Skipping {output_file} - already exists")
                continue
            
            cmd = [
                "python", "-u", "forget_full_class_main.py",
                "-net", exp_config['model'],
                "-weight_path", os.path.join(BASE_PATH, exp_config['baseline_path']),
                "-dataset", exp_config['dataset'],
                "-classes", str(exp_config['num_classes']),
                "-method", method_cmd,
                "-ret_perc", str(ret_perc),
                "-forget_class", exp_config['forget_class'],
                "-seed", str(seed),
                "-output_dir", output_dir,
                "-gpu",
            ]
            retrain_folder = resolve_retrain_folder(exp_config.get("retrain_folder"))
            if retrain_folder:
                cmd.extend(["-retrain_folder", retrain_folder])
            
            print(f"\nRunning: {method_name} | Seed: {seed}", flush=True)
            run_command_realtime(cmd)
    
    # Compile results
    csv_path = run_convert_csv(output_dir)
    if os.path.exists(csv_path):
        run_calculate_miau(csv_path)

## Sub Class Forgetting (veg)

In [ ]:
# ========== SUB CLASS EXPERIMENTS ==========

for exp_config in CONFIG.get('subclass_experiments', []):
    name = exp_config['name']
    output_dir = os.path.join(OUTPUT_BASE, f"Results_{name}")
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n{'#'*80}")
    print(f"# SUB CLASS EXPERIMENT: {name}")
    print(f"# Forgetting class: {exp_config['forget_class']}")
    print(f"{'#'*80}", flush=True)
    
    for seed in SEEDS:
        for method_config in METHOD_CONFIGS:
            # Support both old-style string and new-style dict
            if isinstance(method_config, str):
                method_name = method_config
                method_cmd = method_config
                ret_perc = 0
            else:
                method_name = method_config['name']
                method_cmd = method_config['method']
                ret_perc = method_config.get('ret_perc', 0)
            
            # Check if result file already exists (resume capability)
            output_file = f"{method_name}_{exp_config['dataset'].lower()}_{exp_config['model'].lower()}_seed_{seed}.txt"
            output_path = os.path.join(output_dir, output_file)
            if os.path.exists(output_path):
                print(f"⏭️ Skipping {output_file} - already exists")
                continue
            
            cmd = [
                "python", "-u", "forget_subclass_main.py",
                "-net", exp_config['model'],
                "-weight_path", os.path.join(BASE_PATH, exp_config['baseline_path']),
                "-dataset", exp_config['dataset'],
                "-classes", str(exp_config['num_classes']),
                "-method", method_cmd,
                "-ret_perc", str(ret_perc),
                "-forget_class", exp_config['forget_class'],
                "-seed", str(seed),
                "-my_seed", str(seed),
                "-output_dir", output_dir,
                "-gpu",
            ]
            retrain_folder = resolve_retrain_folder(exp_config.get("retrain_folder"))
            if retrain_folder:
                cmd.extend(["-retrain_folder", retrain_folder])
            
            print(f"\nRunning: {method_name} | Seed: {seed}", flush=True)
            run_command_realtime(cmd)
    
    # Compile results
    csv_path = run_convert_csv(output_dir)
    if os.path.exists(csv_path):
        run_calculate_miau(csv_path)

## Advanced MIA Experiments (main_new_metrics.py)

Run unlearning methods with advanced MIA metrics (LiRA, Quantile, Shadow) using `main_new_metrics.py`.
This is similar to standard experiments but uses `forget_new_metrics.py` for evaluation.

In [ ]:
# ========== ADVANCED MIA EXPERIMENTS ==========
# Uses main_new_metrics.py which runs unlearning methods with advanced MIA metrics
# (LiRA, Quantile, Shadow attacks evaluated through forget_new_metrics.py)

for exp_config in CONFIG.get('advanced_mia_experiments', []):
    name = exp_config['name']
    
    print(f"\n{'#'*80}")
    print(f"# ADVANCED MIA EXPERIMENT: {name}")
    print(f"{'#'*80}")
    
    # Run on fewer seeds since advanced MIA is slower
    advanced_seeds = SEEDS[:3]
    
    for seed in advanced_seeds:
        for method_config in METHOD_CONFIGS:
            try:
                run_main_new_metrics(exp_config, seed, method_config)
            except Exception as e:
                print(f"ERROR: {e}")
    
    # Compile results
    output_folder = os.path.join(OUTPUT_BASE, f"Results_{name}_AdvancedMIA")
    csv_path = run_convert_csv(output_folder)
    if os.path.exists(csv_path):
        run_calculate_miau(csv_path)
    print(f"Advanced MIA results saved to: {csv_path}")

## Summary

In [ ]:
# Print summary of all results
import pandas as pd

print("\n" + "="*80)
print("EXPERIMENT SUMMARY")
print("="*80)

for folder in sorted(os.listdir(OUTPUT_BASE)):
    folder_path = os.path.join(OUTPUT_BASE, folder)
    if os.path.isdir(folder_path):
        miau_file = os.path.join(folder_path, 'compiled_results_MIAU.csv')
        csv_file = os.path.join(folder_path, 'compiled_results.csv')
        
        if os.path.exists(miau_file):
            df = pd.read_csv(miau_file)
            print(f"\n{folder}:")
            print(f"  Total results: {len(df)}")
            if 'MIAU' in df.columns and 'unlearning' in df.columns:
                for method in sorted(df['unlearning'].unique()):
                    method_df = df[df['unlearning'] == method]
                    miau_mean = method_df['MIAU'].mean()
                    miau_std = method_df['MIAU'].std()
                    print(f"  {method}: MIAU = {miau_mean:.2f} ± {miau_std:.2f}")
        elif os.path.exists(csv_file):
            df = pd.read_csv(csv_file)
            print(f"\n{folder}: {len(df)} results (no MIAU calculated)")

print("\n" + "="*80)
print("All experiments completed!")
print(f"Results saved to: {os.path.join(BASE_PATH, OUTPUT_BASE)}")
print("="*80)